# M3 v2 — Juggler with Calibrated L_STAR

## Key fix: proper target calibration
The original M3 set L_STAR = [0.8, 0.1, 0.001] *before* knowing actual validation losses.
From the M3 3B run, initial val losses were cw=1.28, dec=0.087, ent=0.0012.
L_STAR[1]=0.1 was *above* val_dec=0.087 — meaning that loss was already met, so the
Juggler had no incentive to improve it and collapsed its weight.

## Calibration procedure
1. **Run 1 Juggler epoch** with equal weights (α=[1,1,1]) → record val losses
2. **Set L_STAR = 0.85 × val_losses** — 15% below initial, ensuring all losses are "behind target"
3. **Run remaining epochs** with the calibrated targets

This follows the Juggler paper's requirement that L_STAR must be *feasible but challenging*.

In [1]:
# Cell 0: Environment + Model Selection
# ⚠️ RESTART KERNEL between models. One model per session.
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_DEVICE_MAX_CONNECTIONS"] = "1"

import torch
torch.cuda.set_per_process_memory_fraction(0.80)

# TRAIN_MODEL = "qwen25_1p5b_base"
# TRAIN_MODEL = "qwen25_3b_base"
TRAIN_MODEL = "qwen25_7b_base"

use_bf16 = "7b" not in TRAIN_MODEL

print(f"Will train: {TRAIN_MODEL} | bf16={use_bf16}")
print("⚠️ Restart kernel before switching to a different model")

Will train: qwen25_7b_base | bf16=False
⚠️ Restart kernel before switching to a different model


In [2]:
# Cell 1: Imports + data
import os, sys, json, math, re
from typing import List, Dict, Any, Tuple
from tqdm import tqdm

import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import TrainingArguments, Trainer

sys.path.insert(0, os.path.expanduser("~/kd_project"))
from shared_utils import (
    RUNS_DIR, MAX_SEQ_LEN, LR, SEED,
    W_FORMAT, W_DECISION, W_EXPL, LAMBDA,
    STUDENTS, load_data, load_student,
    get_section_spans, in_any_span,
    compute_alpha, compute_mean_confidence,
    find_decision_span_chars, teacher_section_entropy_mean,
    build_prompt_text, tokenize_and_mask, FlexibleCollator,
)

OUT_DIR = os.path.join(RUNS_DIR, "m3v2_juggler")
os.makedirs(OUT_DIR, exist_ok=True)

# Juggler config
N_JUGGLER_EPOCHS = 8
ALPHA_INIT = np.array([1.0, 1.0, 1.0])
LOSS_NAMES = ["L_cw_wsft", "L_dec_only", "L_dec_ent"]
EPSILON_1 = 0.5
T_BASE = 36
R = 5.0
N_VAL = 200

# L_STAR will be AUTO-CALIBRATED after first epoch
L_STAR_DISCOUNT = 0.7  # set targets at 85% of initial val losses

def epsilon_kappa(k): return EPSILON_1 * (k ** -0.95)
def T_kappa(k): return max(1, int(T_BASE * (k ** 0.2)))
def project_R(alpha, R):
    alpha = np.maximum(alpha, 0.0)
    norm = np.linalg.norm(alpha)
    return alpha * (R / norm) if norm > R else alpha

prompts, teacher_rows = load_data()
MEAN_C = compute_mean_confidence(teacher_rows)

# Train/val split for Juggler
import random
rng = random.Random(SEED)
indices = list(range(len(teacher_rows)))
rng.shuffle(indices)
val_indices = set(indices[:N_VAL])
train_rows = [teacher_rows[i] for i in indices if i not in val_indices]
val_rows = [teacher_rows[i] for i in val_indices]
print(f"Juggler train: {len(train_rows)}, val: {len(val_rows)}")

c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loaded 5000 teacher samples from: data\clinical_pharm_teacher_train_6000.jsonl
Juggler train: 4800, val: 200


In [3]:
# Cell 2: Dataset (same as original M3)
class JugglerDatasetFast(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct, mean_c):
        print("  Precomputing dataset...")
        self.items = []
        for idx in tqdm(range(len(rows)), desc="  Tokenizing"):
            r = rows[idx]
            prompt = prompts[r["id"]]
            answer = r["teacher_text"]
            prompt_text = build_prompt_text(tokenizer, prompt, is_instruct)
            input_ids, offsets, labels, answer_start = tokenize_and_mask(tokenizer, prompt_text, answer)

            wsft_weights = [0.0] * len(input_ids)
            spans = get_section_spans(answer)
            dec_spans = [(answer_start+s, answer_start+e) for s,e in spans["decision"]]
            expl_spans = [(answer_start+s, answer_start+e) for s,e in spans["explanation"]]
            for i, (s,e) in enumerate(offsets):
                if e <= s: continue
                if s >= answer_start:
                    w = W_FORMAT
                    if in_any_span(s,e,dec_spans): w = W_DECISION
                    if in_any_span(s,e,expl_spans): w = W_EXPL
                    wsft_weights[i] = float(w)
            active_w = [w for w in wsft_weights if w > 0]
            if active_w:
                mw = sum(active_w)/len(active_w)
                if mw > 1e-6: wsft_weights = [w/mw if w>0 else 0.0 for w in wsft_weights]

            alpha_conf = compute_alpha(r, mean_c)
            dec_mask = torch.zeros(len(input_ids), dtype=torch.bool)
            dec_span_chars = find_decision_span_chars(answer)
            if dec_span_chars:
                ds, de = dec_span_chars
                for i, (s,e) in enumerate(offsets):
                    if labels[i] != -100 and not (e <= answer_start+ds or s >= answer_start+de):
                        dec_mask[i] = True
            ent_teacher = teacher_section_entropy_mean(r, dec_span_chars)

            self.items.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "wsft_weights": torch.tensor(wsft_weights, dtype=torch.float),
                "alpha_conf": torch.tensor(alpha_conf, dtype=torch.float),
                "dec_mask": dec_mask,
                "ent_teacher": ent_teacher.float(),
            })
        print(f"  ✅ Precomputed {len(self.items)} samples")
    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]
print("✅ Dataset ready")

✅ Dataset ready


In [4]:
# Cell 3: Trainer + evaluator (same compute_loss as original, with fixed eval)
class JugglerTrainer(Trainer):
    def __init__(self, juggler_alpha, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.juggler_alpha = juggler_alpha
        self._ce = torch.nn.CrossEntropyLoss(reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        wsft_w = inputs.pop("wsft_weights")
        alpha_conf = inputs.pop("alpha_conf")
        dec_mask = inputs.pop("dec_mask")
        ent_teacher = inputs.pop("ent_teacher")

        out = model(**inputs)
        sl = out.logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        sw = wsft_w[:, 1:].contiguous()
        dm = dec_mask[:, 1:]
        B, S, V = sl.size()

        tl = self._ce(sl.view(-1, V), ll.view(-1)).view(B, S)
        active = (ll != -100).float()

        # L1: CW-WSFT
        w = sw * active
        denom = active.sum(dim=1).clamp(min=1.0)
        L1 = ((tl * w).sum(dim=1) / denom * alpha_conf).mean()

        # L2: Decision-only
        da = dm.float() * active
        L2 = (tl * da).sum() / da.sum().clamp(min=1.0)

        # L3: Decision entropy matching (per-batch loop for memory)
        dec_active = dm & (ll != -100)
        es = torch.zeros(B, device=sl.device)
        for b in range(B):
            idx = dec_active[b].nonzero(as_tuple=True)[0]
            if len(idx) > 0:
                p = torch.softmax(sl[b, idx, :], dim=-1)
                es[b] = -(p * torch.log(p + 1e-9)).sum(-1).mean()
        L3 = LAMBDA * ((es - ent_teacher.to(sl.device)) ** 2).mean()

        a = self.juggler_alpha
        loss = a[0]*L1 + a[1]*L2 + a[2]*L3
        return (loss, out) if return_outputs else loss


@torch.no_grad()
def evaluate_val_losses(model, val_dataset, collator):
    """Returns np.array([L1_mean, L2_mean, L3_mean])."""
    model.eval()
    loader = DataLoader(val_dataset, batch_size=1, collate_fn=collator, shuffle=False)
    sums = np.zeros(3); count = 0
    ce = torch.nn.CrossEntropyLoss(reduction="none")
    for batch in loader:
        batch = {k: v.to("cuda") if hasattr(v,"to") else v for k,v in batch.items()}
        labels=batch.pop("labels"); wsft_w=batch.pop("wsft_weights")
        alpha_conf=batch.pop("alpha_conf"); dm=batch.pop("dec_mask"); et=batch.pop("ent_teacher")
        logits=model(**batch).logits
        sl=logits[:,:-1,:]; ll=labels[:,1:]; sw=wsft_w[:,1:]; dm2=dm[:,1:]
        B,S,V = sl.size()
        tl = ce(sl.reshape(-1,V), ll.reshape(-1)).view(B,S)
        active = (ll != -100).float()
        w = sw*active; denom=active.sum(dim=1).clamp(min=1.0)
        sums[0] += ((tl*w).sum(dim=1)/denom * alpha_conf).sum().item()
        da=dm2.float()*active
        sums[1] += ((tl*da).sum()/da.sum().clamp(min=1.0)).item()
        dec_m = dm2 & (ll!=-100)
        for b in range(B):
            idx=dec_m[b].nonzero(as_tuple=True)[0]
            if len(idx)>0:
                p=torch.softmax(sl[b,idx,:],dim=-1)
                se=-(p*torch.log(p+1e-9)).sum(-1).mean()
                sums[2] += LAMBDA*((se - et[b].to(sl.device))**2).item()
        count += 1
    model.train()
    return sums / max(1, count)

print("✅ Trainer + evaluator ready")

✅ Trainer + evaluator ready


In [ ]:
# Cell 4: Juggler loop with auto-calibrated L_STAR
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc

cfg = STUDENTS[TRAIN_MODEL]
out_path = os.path.join(OUT_DIR, TRAIN_MODEL)
os.makedirs(out_path, exist_ok=True)

RESUME_FROM_KAPPA = 7  # ← set to N+1 to resume after epoch N

alpha_file = os.path.join(out_path, "juggler_state.json")

if RESUME_FROM_KAPPA > 1 and os.path.exists(alpha_file):
    with open(alpha_file) as f:
        saved = json.load(f)
    alpha = np.array(saved["alpha"])
    L_STAR = np.array(saved["L_STAR"])
    history = saved["history"]
    print(f"  Resuming from κ={RESUME_FROM_KAPPA}")
    print(f"  α = {alpha.round(3)}, L_STAR = {L_STAR.round(6)}")

    tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    if "7b" in TRAIN_MODEL:
        from transformers import BitsAndBytesConfig
        bnb = BitsAndBytesConfig(load_in_8bit=True)
        base = AutoModelForCausalLM.from_pretrained(cfg["path"], quantization_config=bnb, device_map="auto", trust_remote_code=True)
    else:
        base = AutoModelForCausalLM.from_pretrained(cfg["path"], torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    model = PeftModel.from_pretrained(base, out_path, is_trainable=True)
    model.train()
else:
    alpha = ALPHA_INIT.copy()
    L_STAR = None  # will be set after first epoch
    history = []
    RESUME_FROM_KAPPA = 1
    tokenizer, model = load_student(TRAIN_MODEL, cfg)

train_ds = JugglerDatasetFast(train_rows, prompts, tokenizer, cfg["is_instruct"], MEAN_C)
val_ds = JugglerDatasetFast(val_rows, prompts, tokenizer, cfg["is_instruct"], MEAN_C)
collator = FlexibleCollator(tokenizer, extra_1d_fields=["wsft_weights", "dec_mask"],
                            extra_scalar_fields=["alpha_conf", "ent_teacher"])

for kappa in range(RESUME_FROM_KAPPA, N_JUGGLER_EPOCHS + 1):
    T_k, eps_k = T_kappa(kappa), epsilon_kappa(kappa)
    if L_STAR is not None:
        print(f"\n--- κ={kappa}: T={T_k}, ε={eps_k:.4f}, α={alpha.round(3)}, L*={L_STAR.round(6)}")
    else:
        print(f"\n--- κ={kappa} (CALIBRATION): T={T_k}, α={alpha.round(3)}")

    trainer = JugglerTrainer(
        juggler_alpha=alpha.tolist(), model=model,
        args=TrainingArguments(
            output_dir=os.path.join(out_path, f"ep{kappa}"), max_steps=T_k,
            per_device_train_batch_size=1, gradient_accumulation_steps=32,
            learning_rate=LR, logging_steps=max(1, T_k//4), save_strategy="no",
            bf16=use_bf16, seed=SEED+kappa, report_to="none",
            remove_unused_columns=False, dataloader_num_workers=0,
        ),
        train_dataset=train_ds, data_collator=collator,
    )
    trainer.train()

    val_losses = evaluate_val_losses(model, val_ds, collator)
    print(f"    val_losses = {dict(zip(LOSS_NAMES, val_losses.round(6)))}")

    # Auto-calibrate L_STAR after first epoch
    if L_STAR is None:
        L_STAR = L_STAR_DISCOUNT * val_losses
        print(f"    ✨ AUTO-CALIBRATED L_STAR = {L_STAR.round(6)} (= {L_STAR_DISCOUNT} × val_losses)")

    delta = val_losses - L_STAR
    alpha_new = project_R(alpha + eps_k * delta, R)
    print(f"    delta = {delta.round(6)}")
    print(f"    α → {alpha_new.round(3)}")

    history.append({"kappa": kappa, "T": T_k, "eps": eps_k,
                     "alpha": alpha.tolist(), "val": val_losses.tolist()})
    alpha = alpha_new

    # Save checkpoint
    model.save_pretrained(out_path)
    tokenizer.save_pretrained(out_path)
    with open(alpha_file, "w") as f:
        json.dump({"alpha": alpha.tolist(), "L_STAR": L_STAR.tolist(), "history": history}, f, indent=2)
    print(f"    ✅ Checkpoint saved")

    del trainer; gc.collect(); torch.cuda.empty_cache()

print(f"\n✅ M3v2 Juggler complete. Final α={alpha.round(3)}")

  Resuming from κ=7
  α = [1.288 1.038 1.   ], L_STAR = [0.791969 0.048902 0.000919]


Loading weights: 100%|██████████| 339/339 [00:14<00:00, 23.10it/s]


  Precomputing dataset...


  Tokenizing: 100%|██████████| 4800/4800 [00:07<00:00, 663.92it/s]


  ✅ Precomputed 4800 samples
  Precomputing dataset...


...

Step,Training Loss
13,38.089989
26,36.486387
39,37.223821
52,36.189711


    val_losses = {'L_cw_wsft': np.float64(0.835449), 'L_dec_only': np.float64(0.058645), 'L_dec_ent': np.float64(0.001199)}
    delta = [0.04348  0.009743 0.00028 ]
    α → [1.291 1.038 1.   ]
    ✅ Checkpoint saved

--- κ=8: T=54, ε=0.0693, α=[1.291 1.038 1.   ], L*=[0.791969 0.048902 0.000919]


Step,Training Loss
13,37.437885
26,35.651978
39,35.423812
52,34.447378


    val_losses = {'L_cw_wsft': np.float64(0.816422), 'L_dec_only': np.float64(0.056876), 'L_dec_ent': np.float64(0.001089)}
    delta = [0.024453 0.007974 0.00017 ]
    α → [1.293 1.039 1.   ]
    ✅ Checkpoint saved

✅ M3v2 Juggler complete. Final α=[1.293 1.039 1.   ]


: 